<h1><center>Agentic AI: Responder and Verifier </center></h1>

In [2]:
import os
import json
import glob
import shutil
import hashlib
import time
import csv
from dataclasses import dataclass, field
from typing import Any, Dict, List
from importlib.metadata import version
from pydantic import BaseModel, Field
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.tools import create_retriever_tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import sys
import atexit
import re
import langchain
import langchain_core
import builtins


# os.environ["OPENAI_API_KEY"] = "xxxxxxxxxxxxx"
with open("OPENAI_API_KEY.txt", "r", encoding="utf-8") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

# === Model selection ===
# PHASE1_MODEL_NAME = "gpt-5-nano-2025-08-07"
# RESPONDER_MODEL_NAME = "gpt-5-nano-2025-08-07"
# VERIFIER_MODEL_NAME = "gpt-5-nano-2025-08-07"
# DEFAULT_TEMPERATURE = 0
# "gpt-5.4-2026-03-05"

OUTPUT_DIR = "outputs"
COT_STEP_RESULTS_PATH = os.path.join(OUTPUT_DIR, "cot_step_results.json")
COT_FINAL_DRAFT_PATH = os.path.join(OUTPUT_DIR, "cot_final_draft.md")
VERIFIED_CONTEXT_PATH = os.path.join(OUTPUT_DIR, "verified_formulation_context.json")
LOG_PATH = os.path.join(OUTPUT_DIR, "log.txt")
COST_LATENCY_TOKEN_DIR = os.path.join(OUTPUT_DIR, "cost_latency_token")
COST_CALL_RECORDS_JSON_PATH = os.path.join(COST_LATENCY_TOKEN_DIR, "call_records.json")
COST_CALL_RECORDS_CSV_PATH = os.path.join(COST_LATENCY_TOKEN_DIR, "call_records.csv")
COST_SUMMARY_JSON_PATH = os.path.join(COST_LATENCY_TOKEN_DIR, "summary.json")

# === Model selection ===
PHASE1_MODEL_NAME = "gpt-5.6-luna"      # "gpt-5.6-sol"       # "gpt-5.6-luna"
RESPONDER_MODEL_NAME = "gpt-5.6-luna"   # "gpt-5.6-sol"    # "gpt-5.6-luna"
VERIFIER_MODEL_NAME = "gpt-5.6-luna"    # "gpt-5.6-sol"     # "gpt-5.6-terra"   #"gpt-5.6-luna"
DEFAULT_TEMPERATURE = 0

# === Reasoning effort selection ===
PHASE1_REASONING_EFFORT = "low"         # Planning: medium
RESPONDER_REASONING_EFFORT = "low"      # Responder: medium
VERIFIER_REASONING_EFFORT = "low"       # "low"   # "medium"    # Verifier: high

# GPT-5.6 系列支持以下等级：
    # none
    # low
    # medium
    # high
    # xhigh
    # max


# Standard API text-token prices in USD per 1M tokens.
# Pricing snapshot date: 2026-07-14.
MODEL_PRICING_USD_PER_1M_TOKENS = {
    "gpt-5.6-sol": {
        "input": 5.00,
        "cached_input": 0.50,
        "cache_creation": 6.25,
        "output": 30.00,
    },
    "gpt-5.6-terra": {
        "input": 2.50,
        "cached_input": 0.25,
        "cache_creation": 3.125,
        "output": 15.00,
    },
    "gpt-5.6-luna": {
        "input": 1.00,
        "cached_input": 0.10,
        "cache_creation": 1.25,
        "output": 6.00,
    },
}
LONG_CONTEXT_INPUT_THRESHOLD = 272_000
LONG_CONTEXT_INPUT_MULTIPLIER = 2.0
LONG_CONTEXT_CACHED_INPUT_MULTIPLIER = 2.0
LONG_CONTEXT_CACHE_CREATION_MULTIPLIER = 2.0
LONG_CONTEXT_OUTPUT_MULTIPLIER = 1.5

selected_models = {
    PHASE1_MODEL_NAME,
    RESPONDER_MODEL_NAME,
    VERIFIER_MODEL_NAME,
}
unpriced_models = sorted(selected_models - set(MODEL_PRICING_USD_PER_1M_TOKENS))
if unpriced_models:
    raise ValueError(
        "Pricing is not configured for the selected model(s): "
        + ", ".join(unpriced_models)
    )

def ensure_output_dir(output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)


def save_text_file(path: str, content: str) -> None:
    ensure_output_dir(os.path.dirname(path) or ".")
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def save_json_file(path: str, data: Any) -> None:
    ensure_output_dir(os.path.dirname(path) or ".")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def initialize_log_file(log_path: str) -> None:
    ensure_output_dir(os.path.dirname(log_path) or ".")
    with open(log_path, "w", encoding="utf-8"):
        pass


def module_print(*args, **kwargs):
    # 正常打印到 notebook / console
    builtins.print(*args, **kwargs)

    # 同时把当前这次 print 的内容写入 log.txt
    sep = kwargs.get("sep", " ")
    end = kwargs.get("end", "\n")
    text = sep.join(str(arg) for arg in args) + end

    ensure_output_dir(os.path.dirname(LOG_PATH) or ".")
    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(text)


initialize_log_file(LOG_PATH)

# 只覆盖当前脚本里的 print，不重定向 sys.stdout / sys.stderr
print = module_print        # 这个行代码之后的日志才会保存在"log.txt"文件中


## Check LangChain version
# from importlib.metadata import version
# print(f'langchain version is: {version("langchain")}')

print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print(f"Log file will be saved to: {LOG_PATH}")


#######################################
# Build or load the RAG database
#######################################

persist_directory = "db_non_circular_v1"
collection_name = "non_circular_rag_v1"
source_directory = "./database"
RAG_FILE_PATTERN = "*.md"
RAG_RETRIEVAL_K = 4
meta_path = os.path.join(persist_directory, "index_meta.json")

splitter_config = {
    "separators": [
        "\n# ",
        "\n## ",
        "\n### ",
        "\n#### ",
        "\n##### ",
        "\n###### ",
        "\n\n",
        "\n",
        " ",
        ""
    ],
    "chunk_size": 1800,
    "chunk_overlap": 300,
    "length_function": len,
    "is_separator_regex": False,
}

embedding_model_name = "text-embedding-ada-002"
embedding = OpenAIEmbeddings(model=embedding_model_name)


def file_sha256(filepath: str) -> str:
    sha = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(chunk)
    return sha.hexdigest()


def build_index_signature(
    source_dir: str,
    file_pattern: str,
    splitter_cfg: dict,
    embedding_model: str,
    collection: str,
):
    source_files = sorted(glob.glob(os.path.join(source_dir, file_pattern)))
    file_records = []

    for path in source_files:
        file_records.append({
            "path": os.path.abspath(path),
            "sha256": file_sha256(path),
        })

    payload = {
        "source_directory": os.path.abspath(source_dir),
        "file_pattern": file_pattern,
        "files": file_records,
        "splitter_config": {
            "separators": splitter_cfg["separators"],
            "chunk_size": splitter_cfg["chunk_size"],
            "chunk_overlap": splitter_cfg["chunk_overlap"],
            "length_function": "len",
            "is_separator_regex": splitter_cfg["is_separator_regex"],
        },
        "embedding_model": embedding_model,
        "collection_name": collection,
        "vector_space": "cosine",
    }

    payload_json = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    signature = hashlib.sha256(payload_json.encode("utf-8")).hexdigest()
    return payload, signature


def should_rebuild_db(persist_dir: str, meta_file: str, current_signature: str) -> tuple[bool, str]:
    if not os.path.isdir(persist_dir):
        return True, f"'{persist_dir}' does not exist. Building a new RAG database."

    if not os.path.isfile(meta_file):
        return True, f"'{meta_file}' does not exist. Rebuilding the RAG database."

    try:
        with open(meta_file, "r", encoding="utf-8") as f:
            saved_meta = json.load(f)
    except Exception as e:
        return True, f"Failed to read index metadata: {e}. Rebuilding the RAG database."

    saved_signature = saved_meta.get("signature")
    if saved_signature != current_signature:
        return True, "Source files or indexing settings changed. Rebuilding the RAG database."

    try:
        test_db = Chroma(
            persist_directory=persist_dir,
            embedding_function=embedding,
            collection_name=collection_name,
        )
        doc_count = test_db._collection.count()
        if doc_count <= 0:
            return True, "Existing Chroma collection is empty. Rebuilding the RAG database."
    except Exception as e:
        return True, f"Failed to load existing Chroma database: {e}. Rebuilding the RAG database."

    return False, f"Found an existing valid RAG database in '{persist_dir}'. Loading it directly."


current_meta_payload, current_signature = build_index_signature(
    source_dir=source_directory,
    file_pattern=RAG_FILE_PATTERN,
    splitter_cfg=splitter_config,
    embedding_model=embedding_model_name,
    collection=collection_name,
)

rebuild_db, rebuild_reason = should_rebuild_db(
    persist_dir=persist_directory,
    meta_file=meta_path,
    current_signature=current_signature,
)
print(rebuild_reason)

if rebuild_db:
    source_files = sorted(glob.glob(os.path.join(source_directory, RAG_FILE_PATTERN)))
    if not source_files:
        raise ValueError(
            f"No files matching '{RAG_FILE_PATTERN}' were found in "
            f"'{source_directory}'. Cannot build the RAG database."
        )

    print("RAG source files to be indexed:")
    for source_file in source_files:
        print(f"  - {source_file}")

    if os.path.isdir(persist_directory):
        shutil.rmtree(persist_directory)

    #######################################
    # Load multiple and process documents
    #######################################
    loader = DirectoryLoader(
        source_directory,
        glob=RAG_FILE_PATTERN,
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    documents = loader.load()

    if not documents:
        raise ValueError(f"No documents were loaded from '{source_directory}'.")

    text_splitter = RecursiveCharacterTextSplitter(
        separators=splitter_config["separators"],
        chunk_size=splitter_config["chunk_size"],
        chunk_overlap=splitter_config["chunk_overlap"],
        length_function=splitter_config["length_function"],
        is_separator_regex=splitter_config["is_separator_regex"],
    )
    texts = text_splitter.split_documents(documents)

    if not texts:
        raise ValueError("Text splitting returned 0 chunks. Cannot build the RAG database.")

    #######################################
    # Create the Database
    #######################################
    vectordb = Chroma.from_documents(
        documents=texts,
        embedding=embedding,
        persist_directory=persist_directory,
        collection_name=collection_name,
        collection_configuration={"hnsw": {"space": "cosine"}},
    )

    os.makedirs(persist_directory, exist_ok=True)
    current_meta_payload["signature"] = current_signature
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(current_meta_payload, f, ensure_ascii=False, indent=2)

    print(f"RAG database built successfully. Number of chunks indexed: {len(texts)}")

else:
    vectordb = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding,
        collection_name=collection_name,
    )

    print(f"Loaded existing RAG database successfully. Number of stored chunks: {vectordb._collection.count()}")


#######################################
# Check the version of Chroma
#######################################
chroma_version = version("langchain-chroma")
print(f"Chroma version: {chroma_version}")


#######################################
# Make independent retrievers with identical settings
#######################################
responder_retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RAG_RETRIEVAL_K}
)

verifier_retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RAG_RETRIEVAL_K}
)

print(f"responder_retriever.search_type is: {responder_retriever.search_type}")
print(f"responder_retriever.search_kwargs is: {responder_retriever.search_kwargs}")
print(f"verifier_retriever.search_type is: {verifier_retriever.search_type}")
print(f"verifier_retriever.search_kwargs is: {verifier_retriever.search_kwargs}")


#######################################
# Two-Agent CoT Pipeline
# Phase 1: Planning  (Responder Agent only)
# Phase 2: Solving   (Responder Agent + Verifier Agent)
#######################################

# === Configuration ===
Switch_Debug = True     # True: save full messages sent to LLM for each step
DEBUG_LOG_DIR = os.path.join(OUTPUT_DIR, "debug_logs")
MAX_RETRIES = 6 # 3

USER_PROMPT_PATH = "User_Prompt.txt"
SYSTEM_PROMPT_1_PATH = "System_Prompt1.txt"
SYSTEM_PROMPT_2_PATH = "System_Prompt2.txt"
SYSTEM_PROMPT_3_PATH = "System_Prompt3.txt"
# GROUND_TRUTH_DIR = "ground_truth"


# === Load Prompt Files ===
def load_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()


system_prompt_1 = load_text_file(SYSTEM_PROMPT_1_PATH)
system_prompt_2 = load_text_file(SYSTEM_PROMPT_2_PATH)
system_prompt_3 = load_text_file(SYSTEM_PROMPT_3_PATH)
user_prompt_full = load_text_file(USER_PROMPT_PATH)

print(f"Loaded System_Prompt1 ({len(system_prompt_1)} chars)")
print(f"Loaded System_Prompt2 ({len(system_prompt_2)} chars)")
print(f"Loaded System_Prompt3 ({len(system_prompt_3)} chars)")
print(f"Loaded User_Prompt    ({len(user_prompt_full)} chars)")


# === Split User Prompt into Part-A and Part-B ===
def split_user_prompt(text: str) -> tuple[str, str]:
    marker = "[Chain-of-Thought Planning]"
    idx = text.find(marker)
    if idx == -1:
        raise ValueError("Cannot find '[Chain-of-Thought Planning]' section in User_Prompt.txt")
    part_a = text[:idx].strip()
    part_b = text[idx:].strip()
    return part_a, part_b


user_prompt_part_a, user_prompt_part_b = split_user_prompt(user_prompt_full)
print(f"User prompt split: Part-A length={len(user_prompt_part_a)}, Part-B length={len(user_prompt_part_b)}")


# === Create independent but identically configured Retriever Tools ===
retriever_document_prompt = PromptTemplate.from_template(
    "Source: {source}\n{page_content}"
)

responder_tool = create_retriever_tool(
    responder_retriever,
    "search_by_similarity",
    (
        "Search the non-circular local corpus for reusable mathematical-modeling "
        "principles. Use focused queries and call this tool before answering."
    ),
    document_prompt=retriever_document_prompt,
    document_separator="\n\n---\n\n",
    response_format="content_and_artifact",
)

verifier_tool = create_retriever_tool(
    verifier_retriever,
    "search_by_similarity",
    (
        "Search the same non-circular local corpus for reusable mathematical-modeling "
        "principles. Use focused queries and call this tool before every judgment."
    ),
    document_prompt=retriever_document_prompt,
    document_separator="\n\n---\n\n",
    response_format="content_and_artifact",
)

responder_tools = [responder_tool]
verifier_tools = [verifier_tool]


# === LLM / Model Selection ===
planning_llm = ChatOpenAI(
    model=PHASE1_MODEL_NAME,
    temperature=DEFAULT_TEMPERATURE,
    reasoning_effort=PHASE1_REASONING_EFFORT
)

responder_llm = ChatOpenAI(
    model=RESPONDER_MODEL_NAME,
    temperature=DEFAULT_TEMPERATURE,
    reasoning_effort=RESPONDER_REASONING_EFFORT,
    use_responses_api=True
)

verifier_llm = ChatOpenAI(
    model=VERIFIER_MODEL_NAME,
    temperature=DEFAULT_TEMPERATURE,
    reasoning_effort=VERIFIER_REASONING_EFFORT,
    use_responses_api=True
)

print(
    f"Phase 1 model: {PHASE1_MODEL_NAME}, "
    f"reasoning effort: {PHASE1_REASONING_EFFORT}"
)
print(
    f"Responder model: {RESPONDER_MODEL_NAME}, "
    f"reasoning effort: {RESPONDER_REASONING_EFFORT}"
)
print(
    f"Verifier model: {VERIFIER_MODEL_NAME}, "
    f"reasoning effort: {VERIFIER_REASONING_EFFORT}"
)
print("Responder API: OpenAI Responses API")
print("Verifier API: OpenAI Responses API")
print(f"Default temperature: {DEFAULT_TEMPERATURE}")


# === Create Agents ===
#
# NOTE: If your langgraph version does not support the `prompt` parameter,
#       replace `prompt=...` with `state_modifier=...` (both accept a string).

# --- Responder Agent (its own RAG tool + its own memory) ---
responder_memory = InMemorySaver()
responder_agent = create_agent(
    model=responder_llm,
    tools=responder_tools,
    system_prompt=system_prompt_2,
    checkpointer=responder_memory,
)
responder_thread = {"configurable": {"thread_id": "responder_solving"}}
print("Responder Agent created (independent RAG tool + memory)")

# --- Verifier Agent (its own RAG tool + independent cross-step memory) ---
verifier_memory = InMemorySaver()
verifier_agent = create_agent(
    model=verifier_llm,
    tools=verifier_tools,
    system_prompt=system_prompt_3,
    checkpointer=verifier_memory,
)
verifier_thread = {"configurable": {"thread_id": "verifier_cross_step"}}
print("Verifier Agent created (independent RAG tool + cross-step memory)")


# === Helper Functions ===

class CostLatencyTokenTracker:
    def __init__(self) -> None:
        self.records: list[dict] = []
        self.history_message_counts: dict[str, int] = {}
        self.pipeline_start_time = time.perf_counter()

    @staticmethod
    def _token_detail(details: dict, key: str) -> int:
        return int(details.get(key, 0) or 0)

    def take_new_messages(self, history_key: str, messages: list) -> list:
        previous_count = self.history_message_counts.get(history_key, 0)
        if len(messages) < previous_count:
            raise ValueError(
                f"Message history for '{history_key}' became shorter unexpectedly."
            )
        new_messages = messages[previous_count:]
        self.history_message_counts[history_key] = len(messages)
        return new_messages

    @staticmethod
    def _message_cost(usage: dict, requested_model: str) -> dict:
        if requested_model not in MODEL_PRICING_USD_PER_1M_TOKENS:
            raise ValueError(
                f"Pricing is not configured for model '{requested_model}'."
            )

        input_tokens = int(usage.get("input_tokens", 0) or 0)
        output_tokens = int(usage.get("output_tokens", 0) or 0)
        total_tokens = int(
            usage.get("total_tokens", input_tokens + output_tokens)
            or (input_tokens + output_tokens)
        )
        input_details = usage.get("input_token_details") or {}
        output_details = usage.get("output_token_details") or {}

        cached_input_tokens = CostLatencyTokenTracker._token_detail(
            input_details, "cache_read"
        )
        cache_creation_input_tokens = CostLatencyTokenTracker._token_detail(
            input_details, "cache_creation"
        )
        reasoning_output_tokens = CostLatencyTokenTracker._token_detail(
            output_details, "reasoning"
        )
        uncached_input_tokens = (
            input_tokens
            - cached_input_tokens
            - cache_creation_input_tokens
        )
        if uncached_input_tokens < 0:
            raise ValueError(
                "Token usage is inconsistent: cache-read/cache-creation tokens "
                "exceed total input tokens."
            )

        long_context = input_tokens > LONG_CONTEXT_INPUT_THRESHOLD
        price = MODEL_PRICING_USD_PER_1M_TOKENS[requested_model]

        input_multiplier = (
            LONG_CONTEXT_INPUT_MULTIPLIER if long_context else 1.0
        )
        cached_input_multiplier = (
            LONG_CONTEXT_CACHED_INPUT_MULTIPLIER if long_context else 1.0
        )
        cache_creation_multiplier = (
            LONG_CONTEXT_CACHE_CREATION_MULTIPLIER if long_context else 1.0
        )
        output_multiplier = (
            LONG_CONTEXT_OUTPUT_MULTIPLIER if long_context else 1.0
        )

        estimated_cost_usd = (
            uncached_input_tokens * price["input"] * input_multiplier
            + cached_input_tokens
            * price["cached_input"]
            * cached_input_multiplier
            + cache_creation_input_tokens
            * price["cache_creation"]
            * cache_creation_multiplier
            + output_tokens * price["output"] * output_multiplier
        ) / 1_000_000

        return {
            "input_tokens": input_tokens,
            "uncached_input_tokens": uncached_input_tokens,
            "cached_input_tokens": cached_input_tokens,
            "cache_creation_input_tokens": cache_creation_input_tokens,
            "output_tokens": output_tokens,
            "reasoning_output_tokens": reasoning_output_tokens,
            "total_tokens": total_tokens,
            "estimated_cost_usd": estimated_cost_usd,
            "long_context_pricing_applied": long_context,
        }

    def record_invocation(
        self,
        role: str,
        requested_model: str,
        new_messages: list,
        latency_seconds: float,
        step_number: int | None = None,
        attempt: int | None = None,
    ) -> dict:
        ai_messages = [
            message
            for message in new_messages
            if getattr(message, "type", "") == "ai"
        ]
        if not ai_messages:
            raise ValueError(
                f"No new AI messages were found for role '{role}'."
            )

        retrieval_call_count = sum(
            1
            for message in new_messages
            if getattr(message, "type", "") == "tool"
        )

        totals = {
            "input_tokens": 0,
            "uncached_input_tokens": 0,
            "cached_input_tokens": 0,
            "cache_creation_input_tokens": 0,
            "output_tokens": 0,
            "reasoning_output_tokens": 0,
            "total_tokens": 0,
            "estimated_cost_usd": 0.0,
            "long_context_request_count": 0,
        }
        response_models = []

        for message in ai_messages:
            usage = getattr(message, "usage_metadata", None)
            if not usage:
                raise ValueError(
                    f"Token usage metadata is missing for role '{role}'."
                )

            message_cost = self._message_cost(usage, requested_model)
            for key in (
                "input_tokens",
                "uncached_input_tokens",
                "cached_input_tokens",
                "cache_creation_input_tokens",
                "output_tokens",
                "reasoning_output_tokens",
                "total_tokens",
            ):
                totals[key] += message_cost[key]
            totals["estimated_cost_usd"] += message_cost["estimated_cost_usd"]
            totals["long_context_request_count"] += int(
                message_cost["long_context_pricing_applied"]
            )

            response_metadata = getattr(message, "response_metadata", {}) or {}
            response_model = response_metadata.get("model_name")
            if response_model:
                response_models.append(str(response_model))

        record = {
            "record_number": len(self.records) + 1,
            "role": role,
            "step_number": step_number,
            "attempt": attempt,
            "requested_model": requested_model,
            "response_models": sorted(set(response_models)),
            "model_call_count": len(ai_messages),
            "retrieval_call_count": retrieval_call_count,
            "latency_seconds": round(float(latency_seconds), 6),
            **totals,
        }
        record["estimated_cost_usd"] = round(
            float(record["estimated_cost_usd"]), 10
        )
        self.records.append(record)
        return record

    def build_summary(self) -> dict:
        grouped: dict[str, dict] = {}
        numeric_fields = (
            "invocation_count",
            "model_call_count",
            "retrieval_call_count",
            "latency_seconds",
            "input_tokens",
            "uncached_input_tokens",
            "cached_input_tokens",
            "cache_creation_input_tokens",
            "output_tokens",
            "reasoning_output_tokens",
            "total_tokens",
            "estimated_cost_usd",
            "long_context_request_count",
        )
        overall = {key: 0 for key in numeric_fields}

        for record in self.records:
            group_key = f"{record['role']}::{record['requested_model']}"
            if group_key not in grouped:
                grouped[group_key] = {
                    "role": record["role"],
                    "requested_model": record["requested_model"],
                    **{key: 0 for key in numeric_fields},
                }

            group = grouped[group_key]
            group["invocation_count"] += 1
            overall["invocation_count"] += 1
            for key in numeric_fields:
                if key == "invocation_count":
                    continue
                group[key] += record[key]
                overall[key] += record[key]

        for aggregate in [*grouped.values(), overall]:
            aggregate["latency_seconds"] = round(
                float(aggregate["latency_seconds"]), 6
            )
            aggregate["estimated_cost_usd"] = round(
                float(aggregate["estimated_cost_usd"]), 10
            )
            invocation_count = int(aggregate["invocation_count"])
            aggregate["average_latency_seconds_per_invocation"] = (
                round(aggregate["latency_seconds"] / invocation_count, 6)
                if invocation_count > 0
                else 0.0
            )

        overall["pipeline_wall_time_seconds"] = round(
            time.perf_counter() - self.pipeline_start_time, 6
        )

        return {
            "pricing_snapshot_date": "2026-07-14",
            "pricing_mode": "OpenAI Standard API",
            "pricing_usd_per_1m_tokens": MODEL_PRICING_USD_PER_1M_TOKENS,
            "long_context_rule": (
                "For each individual model request with more than 272,000 input "
                "tokens, input/cache prices are doubled and output prices are "
                "multiplied by 1.5."
            ),
            "cost_scope": (
                "Planning, Responder, and Verifier text-token costs only. "
                "Embedding/index-construction costs, local Chroma computation, "
                "Batch/Flex/Priority pricing, and regional-processing uplifts are excluded."
            ),
            "groups": list(grouped.values()),
            "overall": overall,
        }

    def save_reports(self) -> dict:
        ensure_output_dir(COST_LATENCY_TOKEN_DIR)
        summary = self.build_summary()
        save_json_file(COST_CALL_RECORDS_JSON_PATH, self.records)
        save_json_file(COST_SUMMARY_JSON_PATH, summary)

        fieldnames = [
            "record_number",
            "role",
            "step_number",
            "attempt",
            "requested_model",
            "response_models",
            "model_call_count",
            "retrieval_call_count",
            "latency_seconds",
            "input_tokens",
            "uncached_input_tokens",
            "cached_input_tokens",
            "cache_creation_input_tokens",
            "output_tokens",
            "reasoning_output_tokens",
            "total_tokens",
            "estimated_cost_usd",
            "long_context_request_count",
        ]
        with open(
            COST_CALL_RECORDS_CSV_PATH,
            "w",
            encoding="utf-8",
            newline="",
        ) as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for record in self.records:
                csv_record = dict(record)
                csv_record["response_models"] = "|".join(
                    record["response_models"]
                )
                writer.writerow(csv_record)

        return summary


usage_tracker = CostLatencyTokenTracker()


def invoke_agent(
    agent,
    message_content: str,
    thread_config: dict,
    role: str,
    requested_model: str,
    history_key: str,
    step_number: int | None = None,
    attempt: int | None = None,
) -> tuple[str, list, list, dict]:
    start_time = time.perf_counter()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": message_content}]},
        thread_config,
    )
    latency_seconds = time.perf_counter() - start_time

    final_message = result["messages"][-1]
    response_text = final_message.text

    if not response_text:
        raise ValueError(
            f"The final AI message for role '{role}' contains no text output."
        )

    new_messages = usage_tracker.take_new_messages(
        history_key=history_key,
        messages=result["messages"],
    )
    usage_record = usage_tracker.record_invocation(
        role=role,
        requested_model=requested_model,
        new_messages=new_messages,
        latency_seconds=latency_seconds,
        step_number=step_number,
        attempt=attempt,
    )
    usage_tracker.save_reports()
    return response_text, result["messages"], new_messages, usage_record


def extract_retrieval_audit(
    new_messages: list,
    require_retrieval: bool = True,
) -> tuple[list[str], list[dict]]:
    retrieval_queries = []
    evidence_records = []

    for message in new_messages:
        if getattr(message, "type", "") == "ai":
            for tool_call in getattr(message, "tool_calls", []) or []:
                if tool_call.get("name") != "search_by_similarity":
                    continue

                query = str(
                    (tool_call.get("args") or {}).get("query", "")
                ).strip()

                if query:
                    retrieval_queries.append(query)

        if getattr(message, "type", "") != "tool":
            continue

        artifact = getattr(message, "artifact", None)

        if artifact:
            documents = artifact if isinstance(artifact, list) else [artifact]

            for document in documents:
                evidence_records.append({
                    "source": str(
                        document.metadata.get(
                            "source",
                            "UNKNOWN_SOURCE",
                        )
                    ),
                    "content": document.page_content.strip(),
                })
        else:
            evidence_records.append({
                "source": "SOURCE_EMBEDDED_IN_TOOL_CONTENT",
                "content": str(
                    getattr(message, "content", "")
                ).strip(),
            })

    if require_retrieval and (
        not retrieval_queries or not evidence_records
    ):
        raise ValueError(
            "The agent did not complete a verifiable RAG retrieval "
            "in this invocation."
        )

    return retrieval_queries, evidence_records


def merge_retrieval_audit(
    accumulated_queries: list[str],
    accumulated_evidence: list[dict],
    new_queries: list[str],
    new_evidence: list[dict],
    retrieved_at_attempt: int,
) -> tuple[list[str], list[dict]]:
    merged_queries = list(accumulated_queries)

    for query in new_queries:
        if query not in merged_queries:
            merged_queries.append(query)

    merged_evidence = [
        dict(record)
        for record in accumulated_evidence
    ]

    seen_evidence = {
        (
            record.get("source", ""),
            record.get("content", ""),
        )
        for record in merged_evidence
    }

    for record in new_evidence:
        annotated_record = dict(record)
        annotated_record["retrieved_at_attempt"] = (
            retrieved_at_attempt
        )

        evidence_key = (
            annotated_record.get("source", ""),
            annotated_record.get("content", ""),
        )

        if evidence_key in seen_evidence:
            continue

        seen_evidence.add(evidence_key)
        merged_evidence.append(annotated_record)

    return merged_queries, merged_evidence


def snapshot_verified_context(context: list[dict]) -> list[dict]:
    return [dict(item) for item in context]


def format_verified_formulation_context(context: list[dict]) -> str:
    if not context:
        return "[No previously verified formulation steps.]"

    sections = []
    for item in context:
        sections.append(
            f"=== Verified Step {item['step_number']} ===\n"
            f"Step query:\n{item['step_query']}\n\n"
            f"Verified answer:\n{item['verified_answer']}"
        )
    return "\n\n".join(sections)


def format_messages_for_debug(messages: list) -> str:
    lines = []
    for i, msg in enumerate(messages):
        msg_type = getattr(msg, "type", "unknown")
        content = getattr(msg, "content", "")
        if isinstance(content, list):
            content = json.dumps(content, ensure_ascii=False, indent=2)
        elif not isinstance(content, str):
            content = str(content)
        lines.append(f"--- Message {i} [{msg_type}] ---\n{content}")
    return "\n\n".join(lines)


def is_verified(verifier_response: str) -> bool:
    return verifier_response.strip().startswith("Your step response is correct")


def build_final_draft_from_step_results(cot_step_results: dict) -> str:
    sections: list[str] = []

    for step in cot_step_results.get("phase2_solving", []):
        if step.get("final_status") != "verified":
            continue

        step_number = step.get("step_number")

        verified_attempts = [
            attempt
            for attempt in step.get("attempts", [])
            if attempt.get("verified") is True
        ]

        if not verified_attempts:
            continue

        verified_answer = verified_attempts[-1].get(
            "responder_output",
            "",
        ).strip()

        if not verified_answer:
            continue

        sections.append(
            f"## CoT Step {step_number}\n\n"
            f"{verified_answer}"
        )

    if not sections:
        return ""

    return (
        "# Final Integrated Mathematical Formulation\n\n"
        + "\n\n".join(sections)
    ).strip()


def validate_markdown_final_draft(draft: str) -> list[str]:
    warnings: list[str] = []

    if not draft.strip():
        warnings.append("The final Markdown draft is empty.")
        return warnings

    forbidden_patterns = {
        r"(?m)^\s*%": (
            "LaTeX comment headings are not allowed."
        ),
        r"(?m)^\s*```": (
            "Fenced code blocks are not allowed in the final draft."
        ),
        (
            r"\\(?:begin|end)\{"
            r"(?:equation\*?|align\*?|split|displaymath|"
            r"gather\*?|multline\*?)"
            r"\}"
        ): (
            "Use `$$...$$`, with `aligned` inside it when needed."
        ),
        r"\\section\{": (
            "Use Markdown headings instead of `\\section`."
        ),
        r"\\subsection\{": (
            "Use Markdown headings instead of `\\subsection`."
        ),
        r"\\paragraph\{": (
            "Use Markdown headings instead of `\\paragraph`."
        ),

        # Match only a standalone LaTeX display opener.
        # Do not match valid line-break spacing such as \\[1mm].
        r"(?<!\\)\\\[": (
            "Use `$$` instead of a standalone `\\[`."
        ),

        # Match only a standalone LaTeX display closer.
        r"(?<!\\)\\\]": (
            "Use `$$` instead of a standalone `\\]`."
        ),
    }

    for pattern, message in forbidden_patterns.items():
        for match in re.finditer(pattern, draft):
            line_number = draft.count(
                "\n",
                0,
                match.start(),
            ) + 1

            warnings.append(
                f"Line {line_number}: {message}"
            )

    if draft.count("$$") % 2 != 0:
        warnings.append(
            "The final Markdown draft contains unmatched `$$` delimiters."
        )

    return sorted(set(warnings))

# === cot_step_results tracking ===
cot_step_results = {
    "pipeline_status": "running",
    "failed_step": None,
    "rag_index": {
        "source_directory": os.path.abspath(source_directory),
        "file_pattern": RAG_FILE_PATTERN,
        "persist_directory": os.path.abspath(persist_directory),
        "collection_name": collection_name,
        "index_signature": current_signature,
        "indexed_files": current_meta_payload["files"],
        "retrieval_method": "similarity",
        "retrieval_k_per_tool_call": RAG_RETRIEVAL_K,
    },
    "model_configuration": {
        "planning_model": PHASE1_MODEL_NAME,
        "responder_model": RESPONDER_MODEL_NAME,
        "verifier_model": VERIFIER_MODEL_NAME,
        "pricing_mode": "OpenAI Standard API",
    },
    "verified_formulation_context": [],
    "phase1_planning": {},
    "phase2_solving": []
}

verified_formulation_context: list[dict] = []
save_json_file(VERIFIED_CONTEXT_PATH, verified_formulation_context)


###############################################################################
# PHASE 1: Planning — Parse CoT steps from User_Prompt
###############################################################################
print("\n" + "=" * 60)
print("PHASE 1: Planning — CoT Step Parsing")
print("=" * 60)

phase1_user_message = (
    f"{user_prompt_full}\n\n"
    "---\n"
    "IMPORTANT: Output your result as a JSON array ONLY, with no additional text, "
    "no markdown fences, and no preamble. Each element must have the following keys:\n"
    '  "step_number": integer,\n'
    '  "step_title": string,\n'
    '  "step_query": string'
)

print("Sending planning request to LLM (direct call, no agent)...")
phase1_start_time = time.perf_counter()
phase1_response = planning_llm.invoke([
    SystemMessage(content=system_prompt_1),
    HumanMessage(content=phase1_user_message),
])
phase1_latency_seconds = time.perf_counter() - phase1_start_time
phase1_usage_record = usage_tracker.record_invocation(
    role="planning",
    requested_model=PHASE1_MODEL_NAME,
    new_messages=[phase1_response],
    latency_seconds=phase1_latency_seconds,
)
usage_tracker.save_reports()
phase1_raw = phase1_response.content
print(f"Phase 1 raw response length: {len(phase1_raw)} chars")

# Parse JSON — strip possible markdown fences
phase1_clean = re.sub(r"^```(?:json)?\s*", "", phase1_raw.strip())
phase1_clean = re.sub(r"\s*```$", "", phase1_clean.strip())

try:
    cot_steps = json.loads(phase1_clean)
except json.JSONDecodeError as e:
    print(f"ERROR: Failed to parse Phase 1 JSON output: {e}")
    print(f"Raw output:\n{phase1_raw}")
    raise

num_steps = len(cot_steps)
print(f"Identified {num_steps} CoT step(s)")

for step in cot_steps:
    step_num = step["step_number"]
    step_path = os.path.join(OUTPUT_DIR, f"CoT_step_query{step_num}.txt")
    save_text_file(step_path, step["step_query"])
    print(f"  Saved: {step_path}")

cot_step_results["phase1_planning"] = {
    "input_system_prompt_file": SYSTEM_PROMPT_1_PATH,
    "input_user_message": phase1_user_message,
    "raw_output": phase1_raw,
    "parsed_steps": cot_steps,
    "num_steps": num_steps,
    "usage": phase1_usage_record,
}

if Switch_Debug:
    ensure_output_dir(DEBUG_LOG_DIR)
    debug_content = (
        f"=== PHASE 1: Planning ===\n\n"
        f"--- System Prompt (from {SYSTEM_PROMPT_1_PATH}) ---\n{system_prompt_1}\n\n"
        f"--- User Message ---\n{phase1_user_message}\n\n"
        f"--- LLM Response ---\n{phase1_raw}\n"
    )
    save_text_file(os.path.join(DEBUG_LOG_DIR, "phase1_planning_full.txt"), debug_content)
    print(f"  Debug log saved: {os.path.join(DEBUG_LOG_DIR, 'phase1_planning_full.txt')}")


###############################################################################
# PHASE 2: Solving — Iterative Responder / Verifier Loop
###############################################################################
print("\n" + "=" * 60)
print("PHASE 2: Solving — Iterative Responder / Verifier Loop")
print("=" * 60)

pipeline_failed = False

for step_idx in range(1, num_steps + 1):
    step_query_path = os.path.join(OUTPUT_DIR, f"CoT_step_query{step_idx}.txt")
    step_query = load_text_file(step_query_path)
    verified_context_before_step = snapshot_verified_context(
        verified_formulation_context
    )
    verified_context_text = format_verified_formulation_context(
        verified_context_before_step
    )

    step_result = {
        "step_number": step_idx,
        "step_query_file": f"CoT_step_query{step_idx}.txt",
        "step_query": step_query,
        "verified_context_before_step": verified_context_before_step,
        "attempts": [],
        "final_status": None,
        "verified_context_after_step": None,
    }

    verified = False
    verifier_feedback = ""

    responder_step_retrieval_queries: list[str] = []
    responder_step_evidence_records: list[dict] = []
    verifier_step_retrieval_queries: list[str] = []
    verifier_step_evidence_records: list[dict] = []

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"\n--- Step {step_idx}, Attempt {attempt} ---")

        attempt_record = {
            "attempt": attempt,
            "responder_input": None,
            "responder_output": None,
            "responder_retrieval_mode": None,
            "responder_new_retrieval_performed": False,
            "responder_retrieval_source_attempts": [],
            "responder_retrieval_queries": [],
            "responder_retrieved_evidence": [],
            "responder_usage": None,
            "verifier_input": None,
            "verifier_output": None,
            "verifier_retrieval_mode": None,
            "verifier_new_retrieval_performed": False,
            "verifier_retrieval_source_attempts": [],
            "verifier_retrieval_queries": [],
            "verifier_retrieved_evidence": [],
            "verifier_usage": None,
            "verified": False,
        }

        # --- 2-1) Responder answers ---
        if attempt == 1:
            responder_msg = (
                f"Use the following system/formulation requirements, current step query, "
                f"and previously verified formulation context. The system/formulation "
                f"requirements and current step query have the highest priority. The "
                f"verified context is authoritative only for previously established "
                f"notation, definitions, and accepted components when it does not conflict "
                f"with those requirements. Before answering, call your "
                f"search_by_similarity tool.\n\n"
                f"=== System and Formulation Requirements ===\n"
                f"{user_prompt_part_a}\n\n"
                f"=== Previously Verified Formulation Context ===\n"
                f"{verified_context_text}\n\n"
                f"=== Current Step Query ===\n"
                f"{step_query}"
            )
        else:
            responder_msg = (
                f"Revise your answer to the same current step. The system/formulation "
                f"requirements and current step query have the highest priority. Use the "
                f"previously verified formulation context to preserve established notation "
                f"and definitions unless it conflicts with an explicit requirement. You may "
                f"reuse the RAG evidence retrieved during earlier attempts of this same step "
                f"because it remains available in your conversation memory. Call your "
                f"search_by_similarity tool again when the Verifier feedback introduces a "
                f"new modeling topic, when the previous evidence is insufficient, or when "
                f"additional focused evidence is needed.\n\n"
                f"=== System and Formulation Requirements ===\n"
                f"{user_prompt_part_a}\n\n"
                f"=== Previously Verified Formulation Context ===\n"
                f"{verified_context_text}\n\n"
                f"=== Current Step Query ===\n"
                f"{step_query}\n\n"
                f"=== Verifier Feedback ===\n"
                f"{verifier_feedback}"
            )

        attempt_record["responder_input"] = responder_msg

        print(f"  Sending to Responder Agent...")
        (
            responder_answer,
            responder_messages,
            responder_new_messages,
            responder_usage_record,
        ) = invoke_agent(
            responder_agent,
            responder_msg,
            responder_thread,
            role="responder",
            requested_model=RESPONDER_MODEL_NAME,
            history_key="responder_solving",
            step_number=step_idx,
            attempt=attempt,
        )
        attempt_record["responder_output"] = responder_answer
        attempt_record["responder_usage"] = responder_usage_record

        answer_path = os.path.join(
            OUTPUT_DIR,
            f"CoT_step_query{step_idx}_answer{attempt}.txt",
        )
        save_text_file(answer_path, responder_answer)
        print(f"  Responder answer saved: {answer_path}")
        print(f"  Responder answer length: {len(responder_answer)} chars")

        (
            current_responder_retrieval_queries,
            current_responder_evidence_records,
        ) = extract_retrieval_audit(
            responder_new_messages,
            require_retrieval=(attempt == 1),
        )

        has_new_queries = bool(
            current_responder_retrieval_queries
        )
        has_new_evidence = bool(
            current_responder_evidence_records
        )

        if has_new_queries != has_new_evidence:
            raise ValueError(
                "Responder retrieval audit is incomplete: retrieval queries "
                "and retrieved evidence must either both be present or both "
                "be absent."
            )

        if has_new_queries and has_new_evidence:
            (
                responder_step_retrieval_queries,
                responder_step_evidence_records,
            ) = merge_retrieval_audit(
                accumulated_queries=responder_step_retrieval_queries,
                accumulated_evidence=responder_step_evidence_records,
                new_queries=current_responder_retrieval_queries,
                new_evidence=current_responder_evidence_records,
                retrieved_at_attempt=attempt,
            )

            responder_retrieval_mode = "new_retrieval"
            responder_new_retrieval_performed = True
        else:
            if (
                not responder_step_retrieval_queries
                or not responder_step_evidence_records
            ):
                raise ValueError(
                    "The Responder did not retrieve RAG evidence for the "
                    "current step, and no evidence from an earlier attempt "
                    "is available for reuse."
                )

            responder_retrieval_mode = (
                "reused_from_previous_attempt"
            )
            responder_new_retrieval_performed = False

        responder_retrieval_source_attempts = sorted({
            int(record["retrieved_at_attempt"])
            for record in responder_step_evidence_records
        })

        attempt_record["responder_retrieval_mode"] = (
            responder_retrieval_mode
        )
        attempt_record["responder_new_retrieval_performed"] = (
            responder_new_retrieval_performed
        )
        attempt_record["responder_retrieval_source_attempts"] = (
            responder_retrieval_source_attempts
        )
        attempt_record["responder_retrieval_queries"] = list(
            responder_step_retrieval_queries
        )
        attempt_record["responder_retrieved_evidence"] = [
            dict(record)
            for record in responder_step_evidence_records
        ]

        responder_retrieval_queries = list(
            responder_step_retrieval_queries
        )
        responder_evidence_records = [
            dict(record)
            for record in responder_step_evidence_records
        ]

        if Switch_Debug:
            ensure_output_dir(DEBUG_LOG_DIR)
            debug_filename = f"CoT_step_query{step_idx}_full_R{attempt}.txt"
            debug_content = (
                f"=== Responder Agent: Step {step_idx}, Attempt {attempt} ===\n\n"
                f"--- Agent System Prompt (from {SYSTEM_PROMPT_2_PATH}) ---\n"
                f"{system_prompt_2}\n\n"
                f"--- User Message Sent This Turn ---\n{responder_msg}\n\n"
                f"--- Responder Retrieval Mode ---\n"
                f"{responder_retrieval_mode}\n\n"
                f"--- New Retrieval Performed This Attempt ---\n"
                f"{responder_new_retrieval_performed}\n\n"
                f"--- Retrieval Source Attempts ---\n"
                f"{json.dumps(responder_retrieval_source_attempts, ensure_ascii=False, indent=2)}\n\n"
                f"--- Retrieval Queries Available for This Answer ---\n"
                f"{json.dumps(responder_retrieval_queries, ensure_ascii=False, indent=2)}\n\n"
                f"--- Retrieved Evidence Available for This Answer ---\n"
                f"{json.dumps(responder_evidence_records, ensure_ascii=False, indent=2)}\n\n"
                f"--- Full Message History (including short-term memory) ---\n"
                f"{format_messages_for_debug(responder_messages)}\n"
            )
            save_text_file(os.path.join(DEBUG_LOG_DIR, debug_filename), debug_content)
            print(f"  Debug log saved: {os.path.join(DEBUG_LOG_DIR, debug_filename)}")

        # --- 2-2) Verifier checks the answer with step-level RAG evidence ---
        if attempt == 1:
            verifier_retrieval_instruction = (
                "This is the first verification attempt for the current step. "
                "Before judging, you MUST call your search_by_similarity tool "
                "at least once. Use multiple focused retrieval queries when "
                "the step contains multiple modeling topics."
            )
        else:
            verifier_retrieval_instruction = (
                "You may reuse the RAG evidence retrieved during earlier "
                "attempts of this same step because it remains available in "
                "your conversation memory. Call your search_by_similarity "
                "tool again when the revised answer introduces new modeling "
                "content, when the previous evidence is insufficient, or "
                "when additional focused evidence is needed."
            )

        verifier_msg = (
            f"Verify the current Responder Answer.\n\n"
            f"{verifier_retrieval_instruction}\n\n"
            f"The System and Formulation Requirements and Current Step Query "
            f"have the highest priority. The explicit Previously Verified "
            f"Formulation Context is the authoritative record only for "
            f"previously established notation, definitions, and accepted "
            f"components when it does not conflict with an explicit "
            f"requirement. Your conversation memory may contain rejected "
            f"drafts; do not treat those rejected drafts as accepted "
            f"content.\n\n"
            f"=== System and Formulation Requirements ===\n"
            f"{user_prompt_part_a}\n\n"
            f"=== Previously Verified Formulation Context ===\n"
            f"{verified_context_text}\n\n"
            f"=== Current Step Query ===\n"
            f"{step_query}\n\n"
            f"=== Current Responder Answer ===\n"
            f"{responder_answer}\n\n"
            f"Provide verification feedback exactly as instructed in your "
            f"system prompt."
        )
        attempt_record["verifier_input"] = verifier_msg

        print(f"  Sending to Verifier Agent...")
        (
            verifier_feedback,
            verifier_messages,
            verifier_new_messages,
            verifier_usage_record,
        ) = invoke_agent(
            verifier_agent,
            verifier_msg,
            verifier_thread,
            role="verifier",
            requested_model=VERIFIER_MODEL_NAME,
            history_key="verifier_cross_step",
            step_number=step_idx,
            attempt=attempt,
        )
        (
            current_verifier_retrieval_queries,
            current_verifier_evidence_records,
        ) = extract_retrieval_audit(
            verifier_new_messages,
            require_retrieval=(attempt == 1),
        )

        has_new_queries = bool(
            current_verifier_retrieval_queries
        )
        has_new_evidence = bool(
            current_verifier_evidence_records
        )

        if has_new_queries != has_new_evidence:
            raise ValueError(
                "Verifier retrieval audit is incomplete: retrieval queries "
                "and retrieved evidence must either both be present or both "
                "be absent."
            )

        if has_new_queries and has_new_evidence:
            (
                verifier_step_retrieval_queries,
                verifier_step_evidence_records,
            ) = merge_retrieval_audit(
                accumulated_queries=verifier_step_retrieval_queries,
                accumulated_evidence=verifier_step_evidence_records,
                new_queries=current_verifier_retrieval_queries,
                new_evidence=current_verifier_evidence_records,
                retrieved_at_attempt=attempt,
            )

            verifier_retrieval_mode = "new_retrieval"
            verifier_new_retrieval_performed = True
        else:
            if (
                not verifier_step_retrieval_queries
                or not verifier_step_evidence_records
            ):
                raise ValueError(
                    "The Verifier did not retrieve RAG evidence for the "
                    "current step, and no evidence from an earlier attempt "
                    "is available for reuse."
                )

            verifier_retrieval_mode = (
                "reused_from_previous_attempt"
            )
            verifier_new_retrieval_performed = False

        verifier_retrieval_source_attempts = sorted({
            int(record["retrieved_at_attempt"])
            for record in verifier_step_evidence_records
        })

        attempt_record["verifier_output"] = verifier_feedback
        attempt_record["verifier_retrieval_mode"] = (
            verifier_retrieval_mode
        )
        attempt_record["verifier_new_retrieval_performed"] = (
            verifier_new_retrieval_performed
        )
        attempt_record["verifier_retrieval_source_attempts"] = (
            verifier_retrieval_source_attempts
        )
        attempt_record["verifier_retrieval_queries"] = list(
            verifier_step_retrieval_queries
        )
        attempt_record["verifier_retrieved_evidence"] = [
            dict(record)
            for record in verifier_step_evidence_records
        ]
        attempt_record["verifier_usage"] = verifier_usage_record

        verifier_retrieval_queries = list(
            verifier_step_retrieval_queries
        )
        verifier_evidence_records = [
            dict(record)
            for record in verifier_step_evidence_records
        ]

        verif_path = os.path.join(OUTPUT_DIR, f"CoT_step_query{step_idx}_verif{attempt}.txt")
        save_text_file(verif_path, verifier_feedback)
        print(f"  Verifier output saved: {verif_path}")

        if Switch_Debug:
            debug_filename_v = f"CoT_step_query{step_idx}_verif_full_R{attempt}.txt"
            debug_content_v = (
                f"=== Verifier Agent: Step {step_idx}, Attempt {attempt} ===\n\n"
                f"--- Agent System Prompt (from {SYSTEM_PROMPT_3_PATH}) ---\n"
                f"{system_prompt_3}\n\n"
                f"--- User Message Sent This Turn ---\n{verifier_msg}\n\n"
                f"--- Verifier Retrieval Mode ---\n"
                f"{verifier_retrieval_mode}\n\n"
                f"--- New Retrieval Performed This Attempt ---\n"
                f"{verifier_new_retrieval_performed}\n\n"
                f"--- Retrieval Source Attempts ---\n"
                f"{json.dumps(verifier_retrieval_source_attempts, ensure_ascii=False, indent=2)}\n\n"
                f"--- Retrieval Queries Available for This Judgment ---\n"
                f"{json.dumps(verifier_retrieval_queries, ensure_ascii=False, indent=2)}\n\n"
                f"--- Retrieved Evidence Available for This Judgment ---\n"
                f"{json.dumps(verifier_evidence_records, ensure_ascii=False, indent=2)}\n\n"
                f"--- Full Message History ---\n"
                f"{format_messages_for_debug(verifier_messages)}\n"
            )
            save_text_file(os.path.join(DEBUG_LOG_DIR, debug_filename_v), debug_content_v)

        # --- 2-4) Check result ---
        if is_verified(verifier_feedback):
            print(f"  VERIFIED: Step {step_idx} passed on attempt {attempt}.")
            verified = True
            attempt_record["verified"] = True
            step_result["attempts"].append(attempt_record)

            verified_formulation_context.append({
                "step_number": step_idx,
                "step_query": step_query,
                "verified_answer": responder_answer,
                "verified_attempt": attempt,
            })
            step_result["verified_context_after_step"] = (
                snapshot_verified_context(verified_formulation_context)
            )
            cot_step_results["verified_formulation_context"] = (
                snapshot_verified_context(verified_formulation_context)
            )
            save_json_file(
                VERIFIED_CONTEXT_PATH,
                verified_formulation_context,
            )
            break
        else:
            print(f"  REJECTED: Step {step_idx} failed on attempt {attempt}.")
            attempt_record["verified"] = False
            step_result["attempts"].append(attempt_record)

    step_result["final_status"] = "verified" if verified else "max_retries_reached"
    if not verified:
        step_result["verified_context_after_step"] = (
            snapshot_verified_context(verified_formulation_context)
        )
        cot_step_results["pipeline_status"] = "stopped_unverified_step"
        cot_step_results["failed_step"] = step_idx
        pipeline_failed = True
        print(
            f"  ERROR: Step {step_idx} reached max retries ({MAX_RETRIES}) "
            f"without verification. Downstream steps will not run."
        )

    cot_step_results["phase2_solving"].append(step_result)
    cot_step_results["verified_formulation_context"] = (
        snapshot_verified_context(verified_formulation_context)
    )
    save_json_file(COT_STEP_RESULTS_PATH, cot_step_results)

    if pipeline_failed:
        break

if not pipeline_failed:
    cot_step_results["pipeline_status"] = "completed"
    cot_step_results["failed_step"] = None

cost_latency_token_summary = usage_tracker.save_reports()
cot_step_results["cost_latency_token_summary"] = cost_latency_token_summary
cot_step_results["verified_formulation_context"] = (
    snapshot_verified_context(verified_formulation_context)
)
save_json_file(COT_STEP_RESULTS_PATH, cot_step_results)
save_json_file(VERIFIED_CONTEXT_PATH, verified_formulation_context)

markdown_warnings: list[str] = []

if cot_step_results["pipeline_status"] == "completed":
    final_draft = build_final_draft_from_step_results(
        cot_step_results
    )
    markdown_warnings = validate_markdown_final_draft(
        final_draft
    )
else:
    final_draft = (
        f"[FINAL DRAFT NOT GENERATED: pipeline stopped because Step "
        f"{cot_step_results['failed_step']} was not verified.]"
    )

# Always save the final draft.
save_text_file(COT_FINAL_DRAFT_PATH, final_draft)

if markdown_warnings:
    print(
        "\nWARNING: The final draft was generated and saved, "
        "but Markdown format checks reported:"
    )

    for warning in markdown_warnings:
        print(f"  - {warning}")

    print(
        "The warnings did not stop final-draft generation."
    )
else:
    print(
        "\nMarkdown format check passed without warnings."
    )

print(f"\nCoT step results saved to: {COT_STEP_RESULTS_PATH}")
print(f"Final integrated draft saved to: {COT_FINAL_DRAFT_PATH}")

print("\n" + "=" * 60)
print("FINAL INTEGRATED DRAFT")
print("=" * 60)
print(final_draft if final_draft else "[EMPTY FINAL DRAFT]")

print("\n" + "=" * 60)
print("Pipeline completed.")
print("=" * 60)

langchain: 1.2.13
langchain_core: 1.2.23
Log file will be saved to: outputs\log.txt
Found an existing valid RAG database in 'db_non_circular_v1'. Loading it directly.
Loaded existing RAG database successfully. Number of stored chunks: 83
Chroma version: 1.1.0
responder_retriever.search_type is: similarity
responder_retriever.search_kwargs is: {'k': 4}
verifier_retriever.search_type is: similarity
verifier_retriever.search_kwargs is: {'k': 4}
Loaded System_Prompt1 (4272 chars)
Loaded System_Prompt2 (10227 chars)
Loaded System_Prompt3 (24684 chars)
Loaded User_Prompt    (31272 chars)
User prompt split: Part-A length=20381, Part-B length=10889
Phase 1 model: gpt-5.6-luna, reasoning effort: low
Responder model: gpt-5.6-luna, reasoning effort: low
Verifier model: gpt-5.6-luna, reasoning effort: low
Responder API: OpenAI Responses API
Verifier API: OpenAI Responses API
Default temperature: 0
Responder Agent created (independent RAG tool + memory)
Verifier Agent created (independent RAG tool 